# Context Delay Metrics (Polars)

Reports robust delay metrics by line, direction, local hour, and weekday/weekend context using the Polars analysis path.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

context_metrics = load_polars_script("polars_context_delay_metrics", "context-delay-metrics.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
LIMIT = 50
MIN_OBSERVATIONS = 50
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"
LINE_REF = None
DIRECTION_REF = None
DAY_TYPE = "all"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    line_ref = LINE_REF
    direction_ref = DIRECTION_REF
    day_type = DAY_TYPE

buckets = context_metrics.load_delay_buckets_for_args(Args)
metrics = context_metrics.build_context_delay_metrics(
    buckets,
    line_ref=LINE_REF,
    direction_ref=DIRECTION_REF,
    day_type=DAY_TYPE,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
metrics


In [ ]:
if metrics.is_empty():
    print("No matching observations found.")
else:
    chart_data = metrics.with_columns(
        (
            pl.col("line_ref")
            + pl.lit(" dir ")
            + pl.col("direction_ref").fill_null("?")
            + pl.lit(" ")
            + pl.col("hour_local")
            + pl.lit(" ")
            + pl.col("day_type")
        ).alias("context")
    )
    fig = px.bar(
        chart_data.sort("p90_delay_min"),
        x="p90_delay_min",
        y="context",
        orientation="h",
        title="Highest-delay line/hour contexts",
        labels={"context": "Context", "p90_delay_min": "P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
